# 2.e) Aggregate Soil Suitability — 0.25° et 1°

Étape d'agrégation, **indépendante de la construction** (`2.c)`). Prend le résultat à résolution
native (une valeur SSR par unité de sol HWSD2, `_soil_suitability_by_smu.pkl`, produit par `2.c)`),
l'applique au raster HWSD2 natif (30 arc-sec) puis moyenne par bloc pour produire les cartes de
production à 0.25° et 1°.

**Méthode d'agrégation : moyenne arithmétique simple**, pas de seuil ni de vote majoritaire — SSR est
une note continue (0-100), pas une classification binaire ; GAEZ lui-même agrège ses résultats sous
forme d'indice continu pondéré par part de classe (voir `GAEZ_v5_Pipeline_Documentation.docx`), pas
par vote majoritaire.

**Sorties** : un fichier NetCDF par culture × régime × résolution, dans
`1_SVM/2_soil_suitability_gaez/soil_suitability_outputs/`. Wheat est dupliqué en
Wheat-spring/Wheat-winter (GAEZ ne distingue pas les deux au niveau du sol).


## 1. Configuration

In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import xarray as xr

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
HWSD_TIF = cu.data_file(CFG, "hwsd_raster")  # data path: see 00_config_and_inventory.ipynb
OUT_DIR = BASE / "1_SVM" / "2_soil_suitability_gaez" / "soil_suitability_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SMU_PKL = OUT_DIR.parent / "_soil_suitability_by_smu.pkl"
if not SMU_PKL.exists():
    raise FileNotFoundError("Run 2.c)_Build_SoilSuitability_HWSD.ipynb first - it produces this file.")
ssr_df = pd.read_pickle(SMU_PKL)
print(ssr_df.shape)

CROP_NAME_ALIASES = {'Wheat': ['Wheat-spring', 'Wheat-winter'], 'Maize': ['Maize'], 'Potato': ['Potato'],
                      'Rice': ['Rice'], 'Soybean': ['Soybean']}
CROPS = ['Wheat', 'Maize', 'Potato', 'Rice', 'Soybean']
# REGIMES reduit a rainfed seul (demande utilisatrice, 2026-10-13, meme choix que 2.c) : le SVM
# n'utilise que des cultures rf. ssr_df ne contient plus que 'rf' depuis le rerun de 2.c) --
# garder ['rf','ir'] ici planterait (sub vide pour 'ir' -> np.nanmax sur un tableau vide).
# Remettre ['rf', 'ir'] si l'irrigue redevient utile (et que 2.c) est re-execute avec).
REGIMES = ['rf']
AGGS = [(30, 0.25), (120, 1.0)]  # (block factor from 30arcsec native HWSD2, output deg)


(144285, 6)


## 2. Agrégation par bloc (moyenne simple)

In [2]:
def build_and_save(smu_ids, values, aggs, crop_label, regime, suffix):
    max_smu = int(np.nanmax(smu_ids))
    lut = np.full(max_smu + 1, np.nan, dtype=np.float32)
    lut[smu_ids.astype(int)] = values.astype(np.float32)
    with rasterio.open(HWSD_TIF) as src:
        W, H = src.width, src.height
        nodata = src.nodata
        for agg_factor, tag in aggs:
            out_w, out_h = W // agg_factor, H // agg_factor
            acc = np.full((out_h, out_w), np.nan, dtype=np.float32)
            for by in range(out_h):
                block = src.read(1, window=Window(0, by * agg_factor, W, agg_factor))
                valid = (block != nodata) & (block <= max_smu)
                vals = np.full(block.shape, np.nan, dtype=np.float32)
                vals[valid] = lut[block[valid]]
                vals_r = vals.reshape(agg_factor, out_w, agg_factor)
                with np.errstate(invalid='ignore'):
                    acc[by, :] = np.nanmean(vals_r, axis=(0, 2))
            da = xr.DataArray(acc, dims=('lat', 'lon'),
                               coords={'lat': np.linspace(90 - tag / 2, -90 + tag / 2, out_h),
                                       'lon': np.linspace(-180 + tag / 2, 180 - tag / 2, out_w)},
                               name=f'soil_suitability_{crop_label}_{regime}',
                               attrs={'source': 'HWSD v2.01 x GAEZ v5 formulas, intermediate input, '
                                                 'by-component architecture (2_soil_suitability_gaez/2.c)',
                                      'crop': crop_label, 'regime': regime, 'input_level': suffix.strip('_') or 'intermediate'})
            for cname in CROP_NAME_ALIASES.get(crop_label, [crop_label]):
                fn = OUT_DIR / f'soil_suitability_{cname}_{regime}{suffix}_{tag}deg.nc'
                da.to_netcdf(fn)
            print(crop_label, regime, suffix or '(intermediate)', tag, 'deg -> valid cells:', int(np.sum(~np.isnan(acc))))


for crop in CROPS:
    for regime in REGIMES:
        sub = ssr_df[(ssr_df['crop'] == crop) & (ssr_df['regime'] == regime)]
        build_and_save(sub['SMU_ID'].values, sub['SSR_int'].values, AGGS, crop, regime, '')

# Pas de Low/High agrege ici : Low/High (Maize/rainfed uniquement) ne servent qu'a la validation
# (2.d), qui les reconstruit elle-meme a la volee depuis _maize_rf_by_smu.pkl -- a resolution
# native (sample_native) pour les statistiques, et a 0.25deg (aggregate_ours_025) seulement pour
# les cartes de comparaison. Aucun fichier agrege ici n'est donc lu par 2.d ; produire des
# fichiers Low/High 0.25deg/1deg en sortie de 2.e serait mort (jamais relu) et hors-sujet, car
# Low/High n'a de sens que pour la validation, pas pour la production (qui est intermediate).
print('DONE')


C:\Users\elisa\AppData\Local\Temp\ipykernel_75124\1925935720.py:18: RuntimeWarning: Mean of empty slice
  acc[by, :] = np.nanmean(vals_r, axis=(0, 2))


Wheat rf (intermediate) 0.25 deg -> valid cells: 252422


C:\Users\elisa\AppData\Local\Temp\ipykernel_75124\1925935720.py:18: RuntimeWarning: Mean of empty slice
  acc[by, :] = np.nanmean(vals_r, axis=(0, 2))


Wheat rf (intermediate) 1.0 deg -> valid cells: 18524


Maize rf (intermediate) 0.25 deg -> valid cells: 252422


...

Potato rf (intermediate) 1.0 deg -> valid cells: 18524


Rice rf (intermediate) 0.25 deg -> valid cells: 252422


Rice rf (intermediate) 1.0 deg -> valid cells: 18524


Soybean rf (intermediate) 0.25 deg -> valid cells: 252422


Soybean rf (intermediate) 1.0 deg -> valid cells: 18524
DONE


## 3. Suite

Ces rasters (production, intermediate, 0.25° et 1°) sont la couche sol à combiner avec le climat au
Module V.


## 4. Sortie 1deg supplementaire -- grille alignee climat/terrain/AMOC (x.0 entier)

**Ajout du 2026-10-02, rien de ci-dessus n'est modifie/supprime.** La sortie `_1.0deg.nc` produite plus haut est sur la grille globale standard (centres `x.5`, ex. 89.5, 88.5...), differente de la grille utilisee par le climat (Module I), le terrain (2.f) et les scenarios AMOC (tous sur des centres entiers `x.0`, confirme sur EC-Earth **et** 20CR-solution). Pour permettre une comparaison/integration directe (Module V) sans re-interpolation, on regenere ici une sortie 1deg **directement depuis le natif HWSD2 (30 arc-sec)**, sur la grille cible x.0 (lat 35..71, lon -25..44 -- la convention deja etablie pour tout le reste du pipeline), plutot que de re-echantillonner le produit `_1.0deg.nc` deja agrege (ce qui ajouterait une double interpolation inutile).

In [3]:
import rasterio
from rasterio.transform import rowcol

# Grille cible (lat 35..71, lon -25..44, centres entiers) ET masque terre 1 deg partage (meme
# fichier que MIRCA2000/ERA5 dans 1_SVM/0_preprocessing_masks/0.1) : Natural Earth 5 arc-min, cellule
# 1deg = terre si >= 25% de ses pixels sont terre. Un seul fichier fournit les deux -- plus besoin
# de dependre de l'ETo du Module I climatique (2_GAEZ, non copie ici) juste pour sa grille, comme
# avant (2026-10-14, simplification demandee par l'utilisatrice : cette copie 1_SVM doit etre
# autonome, sans reference croisee vers 2_GAEZ).
_land_ref = xr.open_dataset(BASE / '1_SVM' / '0_preprocessing_masks' /
                            'outputs_input_data_1deg' / 'land_mask_1deg.nc')
target_lats = _land_ref['lat'].values
target_lons = _land_ref['lon'].values
SHARED_LAND_1DEG = _land_ref['land_mask'].values.astype(bool)
_land_ref.close()
print('grille cible + masque terre (depuis land_mask_1deg.nc) :', target_lats.min(), '-', target_lats.max(),
      '|', target_lons.min(), '-', target_lons.max(), '|', len(target_lats), 'x', len(target_lons))
print('masque terre partage (MIRCA/ERA5) :', int(SHARED_LAND_1DEG.sum()), '/', SHARED_LAND_1DEG.size, 'cellules')

grille cible (depuis climat 1deg) : 35.0 - 71.0 | -25.0 - 44.0 | 37 x 70


masque terre partage (MIRCA/ERA5) : 1332 / 2590 cellules


In [4]:
def build_and_save_x0grid(smu_ids, values, crop_label, regime, level='int'):
    """Meme logique que build_and_save (lookup SMU_ID -> valeur, moyenne par bloc 120x120
    sous-pixels natifs 30 arc-sec = 1deg), mais sur une fenetre geographique precise centree
    sur la grille cible x.0 au lieu d'un decoupage global depuis le pixel (0,0)."""
    max_smu = int(np.nanmax(smu_ids))
    lut = np.full(max_smu + 1, np.nan, dtype=np.float32)
    lut[smu_ids.astype(int)] = values.astype(np.float32)

    agg = 120  # 30 arc-sec x 120 = 1deg
    with rasterio.open(HWSD_TIF) as src:
        nodata = src.nodata
        lat_max, lat_min = target_lats.max() + 0.5, target_lats.min() - 0.5
        lon_min, lon_max = target_lons.min() - 0.5, target_lons.max() + 0.5
        r0, c0 = rowcol(src.transform, lon_min, lat_max)
        h, w = len(target_lats) * agg, len(target_lons) * agg
        block = src.read(1, window=Window(c0, r0, w, h))

    valid = (block != nodata) & (block <= max_smu)
    vals = np.full(block.shape, np.nan, dtype=np.float32)
    vals[valid] = lut[block[valid]]
    vals_r = vals.reshape(len(target_lats), agg, len(target_lons), agg)
    with np.errstate(invalid='ignore'):
        acc = np.nanmean(vals_r, axis=(1, 3))
    # BUG CORRIGE : le raster HWSD est range nord->sud (ligne 0 = lat max), mais target_lats est
    # croissant (sud->nord). Sans ce flip, acc[0] (bloc le plus au nord) se retrouvait associe a
    # target_lats[0] (le plus au sud) -> toute la grille etait inversee nord-sud. Confirme par
    # recoupement : la valeur affichee a tort pour lat=45 correspondait exactement a celle du
    # vrai lat=61 (miroir) dans l'ancienne grille x.5.
    acc = np.flip(acc, axis=0)

    # Meme principe que MIRCA2000/ERA5 (0.1) : une cellule hors du masque terre partage est mise a
    # NaN, meme si HWSD2 y a par ailleurs un peu de couverture (pixels isoles, iles, bords de cote).
    acc = np.where(SHARED_LAND_1DEG, acc, np.nan)

    level_name = {'low': 'low', 'int': 'intermediate', 'high': 'high'}[level]
    da = xr.DataArray(acc, dims=('lat', 'lon'), coords={'lat': target_lats, 'lon': target_lons},
                       name=f'soil_suitability_{crop_label}_{regime}',
                       attrs={'source': f'HWSD v2.01 x GAEZ v5 formulas, {level_name} input, '
                                        'by-component architecture (2_soil_suitability_gaez/2.c), regrille sur '
                                        'la convention x.0 climat/terrain/AMOC',
                              'crop': crop_label, 'regime': regime, 'input_level': level_name})
    # 'int' keeps the original filename (backward compatible with existing consumers, e.g. 5.a);
    # 'low'/'high' get an explicit suffix -- new files, nothing overwritten silently.
    suffix = '' if level == 'int' else f'_{level}'
    for cname in CROP_NAME_ALIASES.get(crop_label, [crop_label]):
        fn = OUT_DIR / f'soil_suitability_{cname}_{regime}_1deg_x0grid{suffix}.nc'
        da.to_netcdf(fn)
    print(crop_label, regime, level, '-> valid cells:', int(np.sum(~np.isnan(acc))), '/', acc.size)


for crop in CROPS:
    for regime in REGIMES:
        sub = ssr_df[(ssr_df['crop'] == crop) & (ssr_df['regime'] == regime)]
        for level, col in [('low', 'SSR_low'), ('int', 'SSR_int'), ('high', 'SSR_high')]:
            build_and_save_x0grid(sub['SMU_ID'].values, sub[col].values, crop, regime, level)
print("DONE -- grille x.0 alignee climat/terrain/AMOC, 3 niveaux d intrant")

C:\Users\elisa\AppData\Local\Temp\ipykernel_75124\2430234182.py:23: RuntimeWarning: Mean of empty slice
  acc = np.nanmean(vals_r, axis=(1, 3))


Wheat rf low -> valid cells: 1330 / 2590


Wheat rf int -> valid cells: 1330 / 2590


Wheat rf high -> valid cells: 1330 / 2590


Maize rf low -> valid cells: 1330 / 2590


...

Rice rf int -> valid cells: 1330 / 2590


Rice rf high -> valid cells: 1330 / 2590


Soybean rf low -> valid cells: 1330 / 2590


Soybean rf int -> valid cells: 1330 / 2590


Soybean rf high -> valid cells: 1330 / 2590
DONE -- grille x.0 alignee climat/terrain/AMOC, 3 niveaux d intrant


## 5. Verification rapide (coherence avec la version x.5 deja existante)

Les deux grilles (x.5 et x.0) ne partagent aucun pixel exact, donc on ne peut pas comparer valeur-a-valeur -- on verifie juste que les deux donnent une image globalement coherente (meme ordre de grandeur, meme pattern spatial) sur un point connu.

In [5]:
# comparaison visuelle grossiere : Maize rf, nord Italie, les deux grilles devraient donner
# une valeur SSR dans le meme ordre de grandeur (pas identique, pixels differents)
old = xr.open_dataset(OUT_DIR / 'soil_suitability_Maize_rf_1.0deg.nc')['soil_suitability_Maize_rf']
new = xr.open_dataset(OUT_DIR / 'soil_suitability_Maize_rf_1deg_x0grid.nc')['soil_suitability_Maize_rf']
print('ancienne grille (x.5), proche Nord Italie :', float(old.sel(lat=45.5, lon=10.5, method='nearest')))
print('nouvelle grille (x.0), proche Nord Italie  :', float(new.sel(lat=45.0, lon=10.0, method='nearest')))

ancienne grille (x.5), proche Nord Italie : 51.49369430541992
nouvelle grille (x.0), proche Nord Italie  : 67.88811492919922


## 6. Sortie 0.25deg supplementaire -- meme grille x.0, alignee sur le climat 1deg (pas la grille x.5 de la section 2)

**Ajout du 2026-10-13, rien ci-dessus n'est modifie/supprime.** Pour combiner le climat 1deg (SVM)
avec le sol a sa vraie resolution fine (au lieu de la moyenne 1deg qui dilue les contraintes dures
comme les glaciers -- voir [[soil-suitability-module-iv-fixes]]), chaque cellule 1deg de la grille
climat x.0 est ici subdivisee en 4x4 = 16 sous-cellules de 0.25deg, chacune avec sa vraie valeur de
sol (pas une moyenne). Le climat sera duplique sur ces 16 sous-cellules (meme valeur, aucune nouvelle
information climatique inventee) dans `5.a.bis)`, seul le sol varie reellement a cette resolution.


In [6]:
def build_and_save_x0grid_fine(smu_ids, values, crop_label, regime, level, res_deg=0.25):
    """Comme build_and_save_x0grid, mais a resolution fine (res_deg degres par cellule de
    sortie), toujours alignee sur la fenetre cible x.0 -- chaque cellule 1deg se subdivise en
    exactement (1/res_deg)^2 sous-cellules, pour combinaison directe avec le climat 1deg par
    duplication (chaque sous-cellule herite du climat de son parent, mais garde sa vraie valeur
    de sol)."""
    max_smu = int(np.nanmax(smu_ids))
    lut = np.full(max_smu + 1, np.nan, dtype=np.float32)
    lut[smu_ids.astype(int)] = values.astype(np.float32)

    sub_per_deg = round(1.0 / res_deg)
    agg = round(120 * res_deg)  # pixels natifs 30 arc-sec par cellule de sortie
    with rasterio.open(HWSD_TIF) as src:
        nodata = src.nodata
        lat_max, lat_min = target_lats.max() + 0.5, target_lats.min() - 0.5
        lon_min, lon_max = target_lons.min() - 0.5, target_lons.max() + 0.5
        r0, c0 = rowcol(src.transform, lon_min, lat_max)
        h, w = len(target_lats) * 120, len(target_lons) * 120  # meme fenetre native que la version 1deg
        block = src.read(1, window=Window(c0, r0, w, h))

    valid = (block != nodata) & (block <= max_smu)
    vals = np.full(block.shape, np.nan, dtype=np.float32)
    vals[valid] = lut[block[valid]]
    nlat_fine = len(target_lats) * sub_per_deg
    nlon_fine = len(target_lons) * sub_per_deg
    vals_r = vals.reshape(nlat_fine, agg, nlon_fine, agg)
    with np.errstate(invalid='ignore'):
        acc = np.nanmean(vals_r, axis=(1, 3))
    # meme correction nord->sud que build_and_save_x0grid (HWSD range nord->sud, target_lats croissant)
    acc = np.flip(acc, axis=0)

    # grille fine : chaque centre 1deg c se subdivise en sub_per_deg sous-centres equidistants
    offsets = (np.arange(sub_per_deg) + 0.5) / sub_per_deg - 0.5  # ex. pour 4 : -0.375,-0.125,0.125,0.375
    fine_lats = (target_lats[:, None] + offsets[None, :]).ravel()
    fine_lons = (target_lons[:, None] + offsets[None, :]).ravel()

    # meme masque terre partage que la version 1deg, repete sur les sous-cellules (coherence avec
    # MIRCA/ERA5 -- pas de nouvelle info terre/mer a cette resolution, juste la meme decision 1deg)
    land_fine = np.repeat(np.repeat(SHARED_LAND_1DEG, sub_per_deg, axis=0), sub_per_deg, axis=1)
    acc = np.where(land_fine, acc, np.nan)

    level_name = {'low': 'low', 'int': 'intermediate', 'high': 'high'}[level]
    da = xr.DataArray(acc, dims=('lat', 'lon'), coords={'lat': fine_lats, 'lon': fine_lons},
                       name=f'soil_suitability_{crop_label}_{regime}',
                       attrs={'source': f'HWSD v2.01 x GAEZ v5 formulas, {level_name} input, '
                                        f'by-component architecture (2_soil_suitability_gaez/2.c), grille {res_deg}deg '
                                        'alignee x.0 (chaque cellule 1deg climat = bloc de sous-cellules)',
                              'crop': crop_label, 'regime': regime, 'input_level': level_name})
    suffix = '' if level == 'int' else f'_{level}'
    for cname in CROP_NAME_ALIASES.get(crop_label, [crop_label]):
        fn = OUT_DIR / f'soil_suitability_{cname}_{regime}_{res_deg}deg_x0grid{suffix}.nc'
        da.to_netcdf(fn)
    print(crop_label, regime, level, f'{res_deg}deg x0grid -> valid cells:', int(np.sum(~np.isnan(acc))), '/', acc.size)


# Restreint aux 4 cultures du SVM (Rice/Soybean non utilises par le SVM climat, pas la peine)
SVM_CROPS = ['Wheat', 'Maize', 'Potato']
for crop in SVM_CROPS:
    for regime in REGIMES:
        sub = ssr_df[(ssr_df['crop'] == crop) & (ssr_df['regime'] == regime)]
        for level, col in [('low', 'SSR_low'), ('int', 'SSR_int'), ('high', 'SSR_high')]:
            build_and_save_x0grid_fine(sub['SMU_ID'].values, sub[col].values, crop, regime, level, res_deg=0.25)
print("DONE -- grille 0.25deg x0grid alignee climat/terrain/AMOC, 3 niveaux d intrant")


C:\Users\elisa\AppData\Local\Temp\ipykernel_75124\456153210.py:28: RuntimeWarning: Mean of empty slice
  acc = np.nanmean(vals_r, axis=(1, 3))


Wheat rf low 0.25deg x0grid -> valid cells: 20409 / 41440


Wheat rf int 0.25deg x0grid -> valid cells: 20409 / 41440


Wheat rf high 0.25deg x0grid -> valid cells: 20409 / 41440


Maize rf low 0.25deg x0grid -> valid cells: 20409 / 41440


Maize rf int 0.25deg x0grid -> valid cells: 20409 / 41440


Maize rf high 0.25deg x0grid -> valid cells: 20409 / 41440


Potato rf low 0.25deg x0grid -> valid cells: 20409 / 41440


Potato rf int 0.25deg x0grid -> valid cells: 20409 / 41440


Potato rf high 0.25deg x0grid -> valid cells: 20409 / 41440
DONE -- grille 0.25deg x0grid alignee climat/terrain/AMOC, 3 niveaux d intrant
